## Gerador de Massa

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from scipy.stats import chi2_contingency, mannwhitneyu, ttest_ind

from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

import statsmodels.api as sm
import statsmodels.formula.api as smf

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)

sns.set_theme(style="whitegrid", context="notebook")

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings('ignore')

# Configurações de visualização
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# 1. CARREGAMENTO DOS DADOS
# Dataset: IBM HR Analytics Employee Attrition & Performance
# Você pode baixar de: https://www.kaggle.com/pavansubhasht/ibm-hr-analytics-attrition-dataset

# Para este exemplo, vou criar um dataset sintético similar
np.random.seed(42)

# Criando dataset sintético para demonstração
n_samples = 1_000_000

data = {
    'Age': np.random.randint(18, 65, n_samples),
    'Attrition': np.random.choice(['Yes', 'No'], n_samples, p=[0.16, 0.84]),
    'BusinessTravel': np.random.choice(['Travel_Rarely', 'Travel_Frequently', 'Non-Travel'], n_samples),
    'DailyRate': np.random.randint(100, 1500, n_samples),
    'Department': np.random.choice(['Sales', 'Research & Development', 'Human Resources'], n_samples),
    'DistanceFromHome': np.random.randint(1, 30, n_samples),
    'Education': np.random.randint(1, 5, n_samples),
    'EducationField': np.random.choice(['Life Sciences', 'Other', 'Medical', 'Marketing', 'Technical Degree', 'Human Resources'], n_samples),
    'EmployeeCount': np.ones(n_samples, dtype=int),
    'EmployeeNumber': np.arange(1, n_samples + 1),
    'EnvironmentSatisfaction': np.random.randint(1, 5, n_samples),
    'Gender': np.random.choice(['Female', 'Male'], n_samples),
    'HourlyRate': np.random.randint(30, 100, n_samples),
    'JobInvolvement': np.random.randint(1, 5, n_samples),
    'JobLevel': np.random.randint(1, 6, n_samples),
    'JobRole': np.random.choice(['Sales Executive', 'Research Scientist', 'Laboratory Technician',
                                'Manufacturing Director', 'Healthcare Representative', 'Manager',
                                'Sales Representative', 'Research Director', 'Human Resources'], n_samples),
    'JobSatisfaction': np.random.randint(1, 5, n_samples),
    'MaritalStatus': np.random.choice(['Single', 'Married', 'Divorced'], n_samples),
    'MonthlyIncome': np.random.randint(1000, 20000, n_samples),
    'MonthlyRate': np.random.randint(2000, 27000, n_samples),
    'NumCompaniesWorked': np.random.randint(0, 10, n_samples),
    'Over18': ['Y'] * n_samples,
    'OverTime': np.random.choice(['Yes', 'No'], n_samples, p=[0.28, 0.72]),
    'PercentSalaryHike': np.random.randint(11, 26, n_samples),
    'PerformanceRating': np.random.choice([3, 4], n_samples, p=[0.84, 0.16]),
    'RelationshipSatisfaction': np.random.randint(1, 5, n_samples),
    'StandardHours': [80] * n_samples,
    'StockOptionLevel': np.random.randint(0, 4, n_samples),
    'TotalWorkingYears': np.random.randint(0, 40, n_samples),
    'TrainingTimesLastYear': np.random.randint(0, 7, n_samples),
    'WorkLifeBalance': np.random.randint(1, 5, n_samples),
    'YearsAtCompany': np.random.randint(0, 40, n_samples),
    'YearsInCurrentRole': np.random.randint(0, 19, n_samples),
    'YearsSinceLastPromotion': np.random.randint(0, 16, n_samples),
    'YearsWithCurrManager': np.random.randint(0, 18, n_samples)
}

df = pd.DataFrame(data)

# Ajustando algumas relações lógicas
df.loc[df['YearsAtCompany'] < df['YearsInCurrentRole'], 'YearsInCurrentRole'] = df['YearsAtCompany']
df.loc[df['YearsAtCompany'] < df['YearsSinceLastPromotion'], 'YearsSinceLastPromotion'] = df['YearsAtCompany']
df.loc[df['TotalWorkingYears'] < df['YearsAtCompany'], 'TotalWorkingYears'] = df['YearsAtCompany']

print("Dataset carregado com sucesso!")
print(f"Dimensões: {df.shape}")
print(f"Colunas: {df.columns.tolist()}")

Dataset carregado com sucesso!
Dimensões: (1000000, 35)
Colunas: ['Age', 'Attrition', 'BusinessTravel', 'DailyRate', 'Department', 'DistanceFromHome', 'Education', 'EducationField', 'EmployeeCount', 'EmployeeNumber', 'EnvironmentSatisfaction', 'Gender', 'HourlyRate', 'JobInvolvement', 'JobLevel', 'JobRole', 'JobSatisfaction', 'MaritalStatus', 'MonthlyIncome', 'MonthlyRate', 'NumCompaniesWorked', 'Over18', 'OverTime', 'PercentSalaryHike', 'PerformanceRating', 'RelationshipSatisfaction', 'StandardHours', 'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance', 'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager']


In [ ]:
# df = generate_ibm_hr_synthetic(n=1470, seed=42)
# df.head()

# df = generate_ibm_hr_synthetic(n=1000000, seed=45)
# df.head()

In [7]:
# Remova a variável oculta antes da análise, para simular uma base real.
df_raw = df.copy()
# df = df.drop(columns=["AttritionProbability_hidden"])

print(df.shape)
df.head()

(1000000, 35)


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,EnvironmentSatisfaction,Gender,HourlyRate,JobInvolvement,JobLevel,JobRole,JobSatisfaction,MaritalStatus,MonthlyIncome,MonthlyRate,NumCompaniesWorked,Over18,OverTime,PercentSalaryHike,PerformanceRating,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,56,No,Travel_Rarely,590,Research & Development,19,1,Other,1,1,4,Female,73,3,3,Research Director,3,Divorced,12378,24652,1,Y,Yes,18,3,1,80,3,33,2,2,29,11,11,1
1,46,No,Travel_Rarely,1441,Human Resources,5,1,Life Sciences,1,2,4,Male,88,4,1,Laboratory Technician,3,Married,9987,15406,2,Y,Yes,12,3,1,80,2,37,3,3,37,6,5,16
2,32,Yes,Travel_Frequently,748,Research & Development,2,3,Medical,1,3,4,Male,89,4,2,Sales Representative,2,Divorced,1276,16073,1,Y,No,15,3,2,80,0,26,1,1,16,3,15,12
3,60,No,Travel_Rarely,1311,Human Resources,3,4,Life Sciences,1,4,2,Female,41,1,4,Research Scientist,2,Single,1807,23260,3,Y,No,17,3,4,80,2,29,3,2,29,17,6,9
4,25,No,Non-Travel,144,Sales,10,4,Life Sciences,1,5,4,Male,66,1,2,Manager,3,Married,14635,2614,1,Y,No,12,3,2,80,1,22,6,4,6,2,6,14


In [8]:
df.shape

(1000000, 35)

In [9]:
nome_arquivo = 'ibm_hr_analytics_sintetico'

df.to_csv(f"raw/{nome_arquivo}.csv", index=False)
print(f"Arquivo salvo em: dados_rh/{nome_arquivo}.csv")

Arquivo salvo em: dados_rh/ibm_hr_analytics_sintetico.csv
